In [1]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=602)
X = dataset.data.features.copy()
y = dataset.data.targets.squeeze()

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

I split the beans into 80% training and 20% test, then using stratify=y so each of the seven varieties keeps the same share in both parts, with random_state=42. I am comparing the models using 5-fold stratified cross-validation on the training data, with the same folds for every model so the comparison is fair. The test set stays untouched until the end.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
log_scores = cross_validate(log_reg, X_train, y_train, cv=cv, return_train_score=True)

print('Training accuracy:', log_scores['train_score'].mean().round(4))
print('Validation accuracy:', log_scores['test_score'].mean().round(4))
print('Average fit time (seconds):', log_scores['fit_time'].mean().round(3))
print('Average scoring time (seconds):', log_scores['score_time'].mean().round(4))

Training accuracy: 0.926
Validation accuracy: 0.9239
Average fit time (seconds): 0.332
Average scoring time (seconds): 0.0072


The logistic regression baseline, is scaled with StandardScaler, has a training accuracy of 0.926 and a validation accuracy of 0.9239. The training took about 0.332 seconds on average and scoring about 0.0072 seconds. The two accuracies are very close, so there is little sign of overfitting. Whether it underfits depends on how the other models score.

In [4]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree_scores = cross_validate(tree, X_train, y_train, cv=cv, return_train_score=True)

print('Training accuracy:', tree_scores['train_score'].mean().round(4))
print('Validation accuracy:', tree_scores['test_score'].mean().round(4))
print('Average fit time (seconds):', tree_scores['fit_time'].mean().round(3))
print('Average scoring time (seconds):', tree_scores['score_time'].mean().round(4))

Training accuracy: 1.0
Validation accuracy: 0.8954
Average fit time (seconds): 0.431
Average scoring time (seconds): 0.0076


The decision tree has a training accuracy of 1.0 but a validation accuracy of 0.8954, so it memorized the training beans and did worse on new ones, which it overfits. It trained in about 0.248 seconds and scored in about 0.0038 seconds.

In [5]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(random_state=42)
forest_scores = cross_validate(forest, X_train, y_train, cv=cv, return_train_score=True)

print('Training accuracy:', forest_scores['train_score'].mean().round(4))
print('Validation accuracy:', forest_scores['test_score'].mean().round(4))
print('Average fit time (seconds):', forest_scores['fit_time'].mean().round(3))
print('Average scoring time (seconds):', forest_scores['score_time'].mean().round(4))

Training accuracy: 1.0
Validation accuracy: 0.9237
Average fit time (seconds): 6.535
Average scoring time (seconds): 0.046


The random forest has a training accuracy of 1.0 and a validation accuracy of 0.9237, which is much better than the single tree but about the same as the logistic regression baseline. It took about 6.535 seconds to train and 0.0.046 seconds to score and this was a lot slower than the baseline.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier

boost = HistGradientBoostingClassifier(random_state=42)
boost_scores = cross_validate(boost, X_train, y_train, cv=cv, return_train_score=True)

print('Training accuracy:', boost_scores['train_score'].mean().round(4))
print('Validation accuracy:', boost_scores['test_score'].mean().round(4))
print('Average fit time (seconds):', boost_scores['fit_time'].mean().round(3))
print('Average scoring time (seconds):', boost_scores['score_time'].mean().round(4))

c:\Users\sergi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\sergi\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\sergi\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\sergi\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Training accuracy: 1.0
Validation accuracy: 0.928
Average fit time (seconds): 5.16
Average scoring time (seconds): 0.0905


The histogram gradient boosting model has a training accuracy of 1.0 and a validation accuracy of 0.928. It took about 5.16 seconds to train and 0.0905 seconds to score. Its validation accuracy was the highest of the four models, just barely higher than the logistic regression baseline.

In [7]:
import pandas as pd

all_scores = {'Logistic regression': log_scores,
              'Decision tree': tree_scores,
              'Random forest': forest_scores,
              'Hist gradient boosting': boost_scores}

rows = []
for name, s in all_scores.items():
    rows.append({'model': name,
                 'train_accuracy': s['train_score'].mean(),
                 'validation_accuracy': s['test_score'].mean(),
                 'gap': s['train_score'].mean() - s['test_score'].mean(),
                 'fit_time_s': s['fit_time'].mean(),
                 'score_time_s': s['score_time'].mean()})

comparison = pd.DataFrame(rows)
print('Cross-validation results on the training data (5 folds)')
print(comparison.round(4).to_string(index=False))

Cross-validation results on the training data (5 folds)
                 model  train_accuracy  validation_accuracy    gap  fit_time_s  score_time_s
   Logistic regression           0.926               0.9239 0.0021      0.3318        0.0072
         Decision tree           1.000               0.8954 0.1046      0.4309        0.0076
         Random forest           1.000               0.9237 0.0763      6.5349        0.0460
Hist gradient boosting           1.000               0.9280 0.0720      5.1599        0.0905


All four models are compared on the same 5 stratified folds of the training data. The logistic regression, random forest and boosting models all have similar validation accuracies of 0.9239, 0.9237 and 0.9280. And the decision tree was lowest at 0.8954. Logistic regression was the fastest to train at 0.3318 seconds and scored in 0.0072 seconds, its training and validation accuracy were almost the same. The boosting model scored highest but took about 5.16 seconds to train and 0.0905 seconds to score a fold of about 2,178 beans. The differences between the top three models are small and I did not measure how much the scores vary between folds.

The decision tree, the random forest and the boosting model all have a training accuracy of 1.0 but lower validation accuracy, with gaps of 0.1046, 0.0763 and 0.0720, so all three overfit the training data. The tree overfit the most and it also has the lowest validation accuracy of 0.8954. Logistic regression has a gap of only 0.0021, with training accuracy 0.926 and validation accuracy 0.9239, so it is not overfitting. Its accuracy of about 92% is not far below the best model’s 0.9280. I don't see anything that says it underfits. I did not test whether regularizing the tree or the ensembles would reduce the gap.

In [8]:
from sklearn.metrics import accuracy_score

log_reg.fit(X_train, y_train)
test_predictions = log_reg.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 4))

Test accuracy: 0.9207


I decided to use the logistic regression with StandardScaler for the production prototype. Its validation accuracy was 0.9239 in cross-validation and its test accuracy was 0.9207. The two agree and there is almost no overfitting with a gap of 0.0021. Boosting scored highest at 0.9280, but that is only about 0.004 higher and boosting took about 3.29 seconds to train, it took longer to score, overfit more and is harder to explain. The logistic regression trained in about 0.332 seconds and scored fast, it is the simplest of the four models to maintain. I did not check which bean varieties it confuses with each other and I did not test how much the scores vary between folds.

In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_scaled = StandardScaler().fit_transform(X)

cluster_rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    cluster_rows.append({'k': k,
                         'inertia': km.inertia_,
                         'silhouette': silhouette_score(X_scaled, km.labels_,
                                                        sample_size=5000, random_state=42)})

cluster_table = pd.DataFrame(cluster_rows)
print('K-means on all 13,611 beans, labels hidden')
print(cluster_table.round(3))

K-means on all 13,611 beans, labels hidden
    k     inertia  silhouette
0   2  129900.034       0.395
1   3   94741.987       0.402
2   4   76474.309       0.339
3   5   61893.644       0.355
4   6   55050.070       0.357
5   7   48811.956       0.310
6   8   45241.726       0.302
7   9   41975.595       0.303
8  10   39810.685       0.263


In [10]:
final_k = 3
km_final = KMeans(n_clusters=final_k, n_init=10, random_state=42).fit(X_scaled)
cluster_labels = km_final.labels_

print('Bean class counts in each k-means cluster (k = 3, labels used only after clustering)')
print(pd.crosstab(cluster_labels, y.values, rownames=['cluster'], colnames=['bean class']))

Bean class counts in each k-means cluster (k = 3, labels used only after clustering)
bean class  BARBUNYA  BOMBAY  CALI  DERMASON  HOROZ  SEKER  SIRA
cluster                                                         
0                108       0     7      3507     19   2019  1919
1                  1     522     1         0      0      0     0
2               1213       0  1622        39   1909      8   717


I ran k-means on the scaled measurements of all 13,611 beans without using the class labels. I chose k = 3 because it had the highest silhouette score (0.402). The silhouette scores were all below 0.5, so the groups are not strongly separated. After clustering, I compared the clusters with the hidden labels. Cluster 1 contained 524 beans and 522 of them are Bombay, which is the whole Bombay class, so the measurements separate that variety clearly. Cluster 0 was mostly Dermason, Seker and Sira beans. Cluster 2 was mostly Barbunya, Cali, Horoz and some Sira beans, these clusters mix several varieties. The Sira beans were split across two clusters. This means the clusters are useful for seeing how the measurements group, but a cluster is not the same as a bean variety and I do not assume that the cluster numbers match the bean classes.

In [11]:
import sys, numpy, pandas, sklearn
from importlib.metadata import version

print('Python', sys.version.split()[0])
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('scikit-learn', sklearn.__version__)
print('ucimlrepo', version('ucimlrepo'))

Python 3.13.5
numpy 2.1.3
pandas 2.2.3
scikit-learn 1.6.1
ucimlrepo 0.0.7
